# Solar Filament Segmentation: train on Kaggle and submit

End-to-end pipeline on Kaggle GPUs: **setup → train the 5 folds → tune post-processing on out-of-fold predictions →
predict the test set → write and validate `submission.csv`.**

All logic lives in the repository ([LucaT2/solar-filament-segmentation](https://github.com/LucaT2/solar-filament-segmentation)),
so this notebook only orchestrates. The same commands run locally (see the README).

**Notebook settings:** Accelerator **GPU T4 ×2** (two folds train in parallel, one per GPU), Internet **on**,
competition data **filament-segmentation-2026** attached.

**Code access** (the repo is private until the deadline), either:
* **Option A:** add a Kaggle *Secret* named `GITHUB_TOKEN` (a GitHub fine-grained token with read access to the
  repo; *Add-ons → Secrets*), or
* **Option B:** upload the repo as a Kaggle Dataset and attach it (set `REPO_DATASET` below).

**Run tracking (optional):** add a Secret named `WANDB_API_KEY` (from wandb.ai/authorize). Each fold then appears as a
run in the W&B project `solar-filament-segmentation`, grouped by experiment, plus one "tune" run with the
post-processing search. Without the secret, training runs normally without tracking.

In [ ]:
# ---- configuration ---------------------------------------------------------------------------------------
CONFIG = "configs/unet_r34.yaml"   # experiment config inside the repo
OVERRIDES = []                      # e.g. ["epochs=20", "num_workers=2"]
FOLDS = [0, 1, 2, 3, 4]             # folds to train in this session ([] = skip training, use attached checkpoints)
TTA = True                          # average 4 flips at test time
REPO_URL = "github.com/LucaT2/solar-filament-segmentation.git"
REPO_BRANCH = "main"
REPO_DATASET = None                 # Option B: e.g. "/kaggle/input/solar-filament-code"
CHECKPOINT_DATASET = None           # optional: dataset with previously trained fold*.pt + oof/ to reuse

## 1. Setup: code and packages

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
try:   # optional W&B tracking; subprocesses inherit the environment variable
    os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
    print("W&B tracking enabled")
except Exception:
    os.environ["WANDB_MODE"] = "disabled"
    print("no WANDB_API_KEY secret, tracking disabled")

REPO = Path("/kaggle/working/repo")
if REPO_DATASET:
    shutil.copytree(REPO_DATASET, REPO, dirs_exist_ok=True)
elif not REPO.exists():
    token = secrets.get_secret("GITHUB_TOKEN")
    subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, f"https://{token}@{REPO_URL}", str(REPO)],
                   check=True, capture_output=True)
print(subprocess.run(["git", "-C", str(REPO), "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# Kaggle images ship torch, numpy, pandas, opencv, scikit-image, timm and wandb. Install only what is missing, pinned
# to the versions tested locally (requirements.txt), without touching the preinstalled stack.
!pip install -q --no-deps segmentation-models-pytorch==0.5.0 albumentations==2.0.8 albucore==0.0.24 pycocotools
!pip install -q --no-deps -e {REPO}
sys.path.insert(0, str(REPO / "src"))
os.chdir(REPO)

import torch
print("torch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
!python scripts/check_env.py

## 2. Train the folds

Each fold trains on the other 4 folds, keeps the checkpoint with the best validation PQ, and writes probability
maps for its validation images to `outputs/<name>/oof/`. Folds are distributed over the available GPUs. A fold takes
roughly 45–60 min on a T4 with the baseline config, so 5 folds on 2 GPUs take about 3 hours.

In [ ]:
from filament.train import TrainConfig
cfg = TrainConfig.load(CONFIG, OVERRIDES)
EXP = Path("/kaggle/working") / cfg.name
print(cfg)

if CHECKPOINT_DATASET:   # reuse earlier sessions' results
    shutil.copytree(CHECKPOINT_DATASET, EXP, dirs_exist_ok=True)

def train_folds_on_gpus(folds):
    n_gpu = max(torch.cuda.device_count(), 1)
    queues = [folds[i::n_gpu] for i in range(n_gpu)]
    procs = []
    for gpu, fold_list in enumerate(queues):
        if not fold_list:
            continue
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "PYTHONUNBUFFERED": "1"}
        cmd = [sys.executable, "-W", "ignore", "scripts/train.py", "--config", CONFIG,
               "--fold", *map(str, fold_list), "--set", *OVERRIDES]
        log = open(f"/kaggle/working/train_gpu{gpu}.log", "w")
        procs.append((subprocess.Popen(cmd, env=env, stdout=log, stderr=subprocess.STDOUT), log, gpu))
        print(f"GPU {gpu}: folds {fold_list}")
    for p, log, gpu in procs:
        p.wait(); log.close()
        print(f"GPU {gpu} finished with exit code {p.returncode}")
        print("".join(open(f"/kaggle/working/train_gpu{gpu}.log").readlines()[-6:]))

if FOLDS:
    train_folds_on_gpus(FOLDS)

In [ ]:
# Training curves: validation PQ at each threshold per epoch
import pandas as pd
for f in sorted(EXP.glob("log_fold*.csv")):
    log = pd.read_csv(f)
    pq_cols = [c for c in log.columns if c.startswith("pq@")]
    best = log[pq_cols].max(axis=1).max()
    print(f"{f.stem}: {len(log)} epochs, best val PQ {best:.4f}, last loss {log.loss.iloc[-1]:.4f}")

## 3. Out-of-fold maps, post-processing, and component rescoring

1. Re-predict every fold's validation images with the same TTA as the test set, so everything tuned on them sees
   identically produced probability maps.
2. Grid-search threshold, minimum area and minimum mean probability on exact OOF PQ → `postprocess.json`.
3. Fit the component rescoring classifier on the OOF components and choose its cut-off on PQ → `rescorer.pkl`
   (applied automatically when writing the submission).

In [ ]:
tta_flag = "--tta" if TTA else ""
!python -W ignore scripts/predict.py --exp {cfg.name} {tta_flag} --split oof
!python -W ignore scripts/tune_postprocess.py --exp {cfg.name} --workers 4 \
    --thresholds 0.4 0.5 0.6 --min-areas 100 200 300 400 500 --min-scores 0
!python -W ignore scripts/fit_rescorer.py --exp {cfg.name}

In [ ]:
# Cross-check: write the OOF predictions as a CSV and score it with the verified metric implementation.
# --no-rescore: the rescorer was fitted on these same OOF components, so scoring OOF with it would be in-sample
# (optimistic). The honest PQ *with* rescoring is the nested estimate in rescorer.json.
import json
trained = sorted(p.stem[4:] for p in EXP.glob("fold*.pt"))
!python -W ignore scripts/make_submission.py --exp {cfg.name} --split oof --no-rescore
!python -W ignore scripts/evaluate.py --pred {EXP}/oof_submission.csv --fold {" ".join(trained)}
print(json.dumps(json.loads((EXP / "rescorer.json").read_text()), indent=2))

## 4. Predict the test set and write the submission

In [ ]:
!python -W ignore scripts/predict.py --exp {cfg.name} {tta_flag}
!python -W ignore scripts/make_submission.py --exp {cfg.name}
shutil.copy(EXP / "submission.csv", "/kaggle/working/submission.csv")

In [ ]:
from filament.paths import get_data_paths
from filament.submission import read_submission, validate_submission
sub = read_submission("/kaggle/working/submission.csv")
test_stems = [p.stem for p in get_data_paths().test_images.glob("*.jpeg")]
print(len(sub), "filaments,", sub.filament_id.str.rsplit("_", n=1).str[0].nunique(), "of", len(test_stems), "images")
print("validation:", validate_submission(sub, expected_stems=test_stems) or "OK")
sub.head()

## 5. Submit

`/kaggle/working/submission.csv` is in this notebook's output. Download it and upload it on the competition page
(*Submit Prediction*), or from a machine with Kaggle credentials:

```
kaggle competitions submit -c filament-segmentation-2026 -f submission.csv -m "unet_r34 5-fold + TTA"
```

To reuse the trained models in a later session, save this notebook's output as a Dataset and set
`CHECKPOINT_DATASET` (with `FOLDS = []` to skip training).